<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Variant_Calling_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(90deg, #0F2027, #203A43, #2C5364); padding: 30px 25px; border-radius: 14px; margin-bottom: 10px;">
  <h1 style="color: #ffffff; font-family: 'Segoe UI', sans-serif; font-size: 34px; font-weight: 800; margin: 0;">
     Variant Calling &amp; Pathogenicity Prediction Pipeline
  </h1>
  <p style="color: #d0e6f0; font-family: 'Segoe UI', sans-serif; font-size: 16px; margin-top: 8px;">
    <b>Project 3 /b> &nbsp;|&nbsp; Bioinformatics &nbsp;|&nbsp; Genomic Variant Analysis + ML Classification
  </p>
</div>


## 📋 Pipeline Overview

| Stage | Description |
|---|---|
| **1. Reference & Reads** | Simulate a reference genome region + sequencing reads |
| **2. Variant Calling** | Detect SNPs/Indels by comparing reads to reference (pileup-based) |
| **3. Annotation** | Add gene, allele frequency, quality, depth, conservation score |
| **4. Visualization** | Interactive Plotly plots — Manhattan plot, variant spectrum, QC plots |
| **5. ML Classification** | Predict variant **Pathogenic vs Benign** from variant features |
| **6. Runtime Prediction** | Apna khud ka variant data daal kar direct predict karein |



# 1. Setup &amp; Imports</b>
</div>


In [1]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


# 2. Simulate Reference Genome &amp; Sequencing Reads</b>
</div>


In [2]:
def simulate_reference(length=2000, seed=1):
    rng = np.random.default_rng(seed)
    bases = np.array(list("ACGT"))
    return "".join(rng.choice(bases, size=length))

reference_seq = simulate_reference(length=2000)
print(f"Reference length: {len(reference_seq)} bp")
print(f"First 80 bp: {reference_seq[:80]}...")


Reference length: 2000 bp
First 80 bp: CGTTAATTACTCCTCCGGAATTTGTCCTACACTACCTAGCATACCCATGTAGCGTCGACTCGCACGCTCGTTCAGGTCCA...


In [3]:
def simulate_reads(reference, n_reads=400, read_length=100, error_rate=0.01, true_variants=None, seed=2):
    """Simulate sequencing reads from the reference, injecting true variants + sequencing errors."""
    rng = np.random.default_rng(seed)
    bases = np.array(list("ACGT"))
    ref_len = len(reference)
    reads = []

    for _ in range(n_reads):
        start = rng.integers(0, ref_len - read_length)
        read = list(reference[start:start + read_length])

        # Inject true variants that fall within this read
        if true_variants:
            for pos, alt in true_variants.items():
                if start <= pos < start + read_length:
                    read[pos - start] = alt

        # Sequencing errors
        for i in range(len(read)):
            if rng.random() < error_rate:
                read[i] = rng.choice(bases)

        reads.append({"start": start, "seq": "".join(read)})

    return reads

# Ground-truth variants (position -> alt allele), some at low allele frequency (heterozygous / subclonal)
true_variant_positions = sorted(np.random.default_rng(3).choice(range(50, 1950), size=25, replace=False))
true_variants = {}
for pos in true_variant_positions:
    ref_base = reference_seq[pos]
    alt_base = np.random.default_rng(pos).choice([b for b in "ACGT" if b != ref_base])
    true_variants[pos] = alt_base

reads = simulate_reads(reference_seq, n_reads=500, read_length=100, true_variants=true_variants)
print(f"Simulated {len(reads)} reads covering {len(reference_seq)} bp reference")


Simulated 500 reads covering 2000 bp reference


# 3. Variant Calling (Pileup-based)</b>


In [4]:
def build_pileup(reads, ref_len):
    pileup = [[] for _ in range(ref_len)]
    for r in reads:
        for i, base in enumerate(r["seq"]):
            pos = r["start"] + i
            pileup[pos].append(base)
    return pileup

pileup = build_pileup(reads, len(reference_seq))

def call_variants(pileup, reference, min_depth=8, min_alt_freq=0.15, min_alt_count=3):
    calls = []
    for pos, bases in enumerate(pileup):
        depth = len(bases)
        if depth < min_depth:
            continue
        ref_base = reference[pos]
        counts = pd.Series(bases).value_counts()
        for alt_base, alt_count in counts.items():
            if alt_base == ref_base:
                continue
            alt_freq = alt_count / depth
            if alt_freq >= min_alt_freq and alt_count >= min_alt_count:
                calls.append({
                    "position": pos, "ref": ref_base, "alt": alt_base,
                    "depth": depth, "alt_count": alt_count, "allele_freq": round(alt_freq, 3)
                })
    return pd.DataFrame(calls)

called_variants = call_variants(pileup, reference_seq)
print(f"Variants called: {len(called_variants)}  (true variants injected: {len(true_variants)})")
called_variants.head(10)


Variants called: 25  (true variants injected: 25)


,position,ref,alt,depth,alt_count,allele_freq
0,111,A,G,33,33,1.000
1,124,T,G,30,30,1.000
2,210,C,A,20,20,1.000
3,227,T,C,18,18,1.000
4,265,T,G,18,18,1.000
5,352,A,C,30,29,0.967
6,386,T,C,34,34,1.000
7,390,A,G,34,34,1.000
8,494,G,C,28,27,0.964
9,550,G,T,30,30,1.000


In [5]:
# Quality score = Phred-like score derived from allele freq + depth confidence (simulated)
called_variants["quality"] = (
    10 * np.log10(called_variants["depth"] * called_variants["allele_freq"] + 1) * 3.3
).round(1)

# True positive check
called_variants["is_true_variant"] = called_variants["position"].isin(true_variants.keys())
print(f"True positives: {called_variants['is_true_variant'].sum()} / {len(true_variants)} true variants recovered")
called_variants.sort_values('quality', ascending=False).head(10)


True positives: 25 / 25 true variants recovered


,position,ref,alt,depth,alt_count,allele_freq,quality,is_true_variant
14,907,T,C,38,38,1.000,52.5,True
6,386,T,C,34,34,1.000,51.0,True
7,390,A,G,34,34,1.000,51.0,True
0,111,A,G,33,33,1.000,50.5,True
15,954,G,T,32,32,1.000,50.1,True
17,1146,T,A,30,30,1.000,49.2,True
9,550,G,T,30,30,1.000,49.2,True
1,124,T,G,30,30,1.000,49.2,True
5,352,A,C,30,29,0.967,48.7,True
16,1031,G,T,29,28,0.966,48.3,True


# 4. Annotate Variants (Gene, Consequence, Conservation, Clinical Label)</b>
</div>



In [6]:
genes = ["BRCA1", "BRCA2", "TP53", "EGFR", "KRAS", "PTEN", "APC", "MLH1"]
consequences = ["missense", "synonymous", "nonsense", "frameshift", "splice_site", "intronic"]
conseq_weight = {"missense": 0.5, "synonymous": 0.03, "nonsense": 0.85, "frameshift": 0.9, "splice_site": 0.7, "intronic": 0.05}

n = len(called_variants)
rng = np.random.default_rng(7)

called_variants["gene"] = rng.choice(genes, size=n)
called_variants["consequence"] = rng.choice(consequences, size=n, p=[0.35, 0.30, 0.08, 0.07, 0.10, 0.10])
called_variants["gnomAD_freq"] = np.round(rng.beta(0.4, 8, size=n), 5)  # population allele frequency
called_variants["conservation_score"] = np.round(rng.uniform(0, 1, size=n), 3)  # GERP/phyloP-like, 0-1

# Simulated pathogenicity label — driven by consequence severity, rarity, and conservation (realistic heuristic)
conseq_score = called_variants["consequence"].map(conseq_weight)
pathogenic_prob = (
    0.55 * conseq_score +
    0.25 * called_variants["conservation_score"] +
    0.20 * (1 - called_variants["gnomAD_freq"] * 20).clip(0, 1)
)
called_variants["pathogenic"] = (rng.random(n) < pathogenic_prob).astype(int)
called_variants["classification"] = called_variants["pathogenic"].map({1: "Pathogenic", 0: "Benign"})

print(f"Pathogenic: {called_variants['pathogenic'].sum()}  |  Benign: {(called_variants['pathogenic']==0).sum()}")
called_variants.head(10)


Pathogenic: 9  |  Benign: 16


,position,ref,alt,depth,alt_count,allele_freq,quality,is_true_variant,gene,consequence,gnomAD_freq,conservation_score,pathogenic,classification
0,111,A,G,33,33,1.000,50.5,True,MLH1,synonymous,0.10483,0.743,0,Benign
1,124,T,G,30,30,1.000,49.2,True,PTEN,synonymous,0.02301,0.581,0,Benign
2,210,C,A,20,20,1.000,43.6,True,PTEN,synonymous,0.00032,0.427,0,Benign
3,227,T,C,18,18,1.000,42.2,True,MLH1,intronic,0.01607,0.878,1,Pathogenic
4,265,T,G,18,18,1.000,42.2,True,KRAS,frameshift,0.00607,0.412,0,Benign
5,352,A,C,30,29,0.967,48.7,True,APC,synonymous,0.04226,0.923,0,Benign
6,386,T,C,34,34,1.000,51.0,True,APC,intronic,0.01771,0.069,0,Benign
7,390,A,G,34,34,1.000,51.0,True,BRCA2,missense,0.36644,0.430,0,Benign
8,494,G,C,28,27,0.964,47.8,True,BRCA1,missense,0.13583,0.520,1,Pathogenic
9,550,G,T,30,30,1.000,49.2,True,TP53,synonymous,0.25510,0.951,1,Pathogenic


# 5. Interactive Visualizations
</div>


In [7]:
# Manhattan-style plot: position vs quality, colored by classification
fig = px.scatter(
    called_variants, x="position", y="quality", color="classification",
    hover_data=["gene", "ref", "alt", "consequence", "allele_freq"],
    title="Variant Quality Across Genomic Positions (Manhattan-style Plot)",
    labels={"position": "Genomic Position (bp)", "quality": "Quality Score"},
    template="plotly_white",
    color_discrete_map={"Pathogenic": "#E63946", "Benign": "#2E86AB"},
    opacity=0.8
)
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))
fig.update_layout(height=550)
fig.show()


In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Variant Type (Consequence)", "Classification Split"),
                     specs=[[{"type": "domain"}, {"type": "domain"}]])

conseq_counts = called_variants["consequence"].value_counts()
fig.add_trace(go.Pie(labels=conseq_counts.index, values=conseq_counts.values, hole=0.45,
                      marker=dict(colors=px.colors.qualitative.Set2)), row=1, col=1)

class_counts = called_variants["classification"].value_counts()
fig.add_trace(go.Pie(labels=class_counts.index, values=class_counts.values, hole=0.45,
                      marker=dict(colors=["#E63946", "#2E86AB"])), row=1, col=2)

fig.update_layout(title_text="Variant Spectrum Overview", height=450)
fig.show()


In [9]:
fig = px.scatter(
    called_variants, x="gnomAD_freq", y="conservation_score", color="classification",
    size="quality", hover_data=["gene", "consequence"],
    title="Allele Frequency vs Conservation Score (bubble size = quality)",
    labels={"gnomAD_freq": "Population Allele Frequency (gnomAD)", "conservation_score": "Conservation Score"},
    template="plotly_white",
    color_discrete_map={"Pathogenic": "#E63946", "Benign": "#2E86AB"}
)
fig.update_layout(height=550)
fig.show()

fig2 = px.box(
    called_variants, x="gene", y="quality", color="classification",
    title="Variant Quality Distribution by Gene",
    template="plotly_white",
    color_discrete_map={"Pathogenic": "#E63946", "Benign": "#2E86AB"}
)
fig2.update_layout(height=500)
fig2.show()


# < 6. Train Pathogenicity Classifier</b>
</div>


In [10]:
feature_cols_numeric = ["depth", "allele_freq", "quality", "gnomAD_freq", "conservation_score"]
feature_cols_categorical = ["consequence", "gene"]

encoders = {}
X_cat = pd.DataFrame(index=called_variants.index)
for col in feature_cols_categorical:
    le = LabelEncoder()
    X_cat[col + "_enc"] = le.fit_transform(called_variants[col])
    encoders[col] = le

X = pd.concat([called_variants[feature_cols_numeric], X_cat], axis=1)
y = called_variants["pathogenic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

variant_scaler = StandardScaler()
X_train_scaled = variant_scaler.fit_transform(X_train)
X_test_scaled = variant_scaler.transform(X_test)

variant_clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
variant_clf.fit(X_train_scaled, y_train)

preds = variant_clf.predict(X_test_scaled)
probs = variant_clf.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"ROC-AUC : {roc_auc_score(y_test, probs):.3f}")
print("\n", classification_report(y_test, preds, target_names=["Benign", "Pathogenic"]))


Accuracy: 0.714
ROC-AUC : 0.583

               precision    recall  f1-score   support

      Benign       0.67      1.00      0.80         4
  Pathogenic       1.00      0.33      0.50         3

    accuracy                           0.71         7
   macro avg       0.83      0.67      0.65         7
weighted avg       0.81      0.71      0.67         7



In [11]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    x=["Benign", "Pathogenic"], y=["Benign", "Pathogenic"],
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Confusion Matrix — Pathogenicity Classifier"
)
fig.update_layout(height=450, width=500)
fig.show()

fpr, tpr, _ = roc_curve(y_test, probs)
auc = roc_auc_score(y_test, probs)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={auc:.3f})', line=dict(color="#E63946", width=3)))
fig2.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(dash='dash', color='gray')))
fig2.update_layout(title="ROC Curve", xaxis_title="False Positive Rate", yaxis_title="True Positive Rate",
                    template="plotly_white", height=500)
fig2.show()

importances = pd.Series(variant_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig3 = px.bar(importances, orientation='h', title="Feature Importance — Pathogenicity Prediction",
              labels={"value": "Importance", "index": "Feature"}, template="plotly_white",
              color=importances.values, color_continuous_scale="Viridis")
fig3.update_layout(height=450, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig3.show()


# 7. Runtime Prediction — Apna Variant Direct Input Karein</b>
</div>



In [12]:
gene_options = genes
consequence_options = consequences

position_box = widgets.IntText(value=500, description="Position (bp)", style={'description_width': '150px'}, layout=widgets.Layout(width='300px'))
ref_box = widgets.Dropdown(options=list("ACGT"), value="A", description="Ref Allele", style={'description_width': '150px'}, layout=widgets.Layout(width='300px'))
alt_box = widgets.Dropdown(options=list("ACGT"), value="G", description="Alt Allele", style={'description_width': '150px'}, layout=widgets.Layout(width='300px'))
gene_box = widgets.Dropdown(options=gene_options, value=gene_options[0], description="Gene", style={'description_width': '150px'}, layout=widgets.Layout(width='300px'))
conseq_box = widgets.Dropdown(options=consequence_options, value="missense", description="Consequence", style={'description_width': '150px'}, layout=widgets.Layout(width='300px'))
depth_box = widgets.IntSlider(value=40, min=8, max=200, description="Read Depth", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
altfreq_box = widgets.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.01, description="Allele Freq (VAF)", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
quality_box = widgets.FloatSlider(value=60.0, min=0.0, max=100.0, step=0.5, description="Quality Score", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
gnomad_box = widgets.FloatSlider(value=0.001, min=0.0, max=0.5, step=0.001, readout_format='.3f', description="gnomAD Freq", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
conservation_box = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.01, description="Conservation Score", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))

predict_btn = widgets.Button(description=" Variant Predict Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='42px'))
out2 = widgets.Output()

def render_variant_result(label, proba, details):
    color = "#E63946" if label == "Pathogenic" else "#2E86AB"
    emoji = "" if label == "Pathogenic" else ""
    conf = proba[1]*100 if label == "Pathogenic" else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font-weight:700; color:{color};">{emoji} Classification: {label}</div>
        <div style="font-size:14px; color:#444; margin-top:4px;">{details}</div>
        <div style="font-size:15px; margin-top:8px;">Confidence: <b>{conf:.2f}%</b></div>
        <div style="margin-top:10px; height:14px; width:100%; background:#e0e0e0; border-radius:7px; overflow:hidden;">
            <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#2E86AB,#E63946);"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>P(Benign) = {proba[0]*100:.2f}%</span><span>P(Pathogenic) = {proba[1]*100:.2f}%</span>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict_variant(b):
    with out2:
        clear_output()
        row = {
            "depth": depth_box.value, "allele_freq": altfreq_box.value, "quality": quality_box.value,
            "gnomAD_freq": gnomad_box.value, "conservation_score": conservation_box.value,
            "consequence_enc": encoders["consequence"].transform([conseq_box.value])[0],
            "gene_enc": encoders["gene"].transform([gene_box.value])[0],
        }
        row_df = pd.DataFrame([row])[feature_cols_numeric + [c + "_enc" for c in feature_cols_categorical]]
        row_scaled = variant_scaler.transform(row_df)
        pred = variant_clf.predict(row_scaled)[0]
        proba = variant_clf.predict_proba(row_scaled)[0]
        label = "Pathogenic" if pred == 1 else "Benign"
        details = f"Position {position_box.value} | {ref_box.value}&gt;{alt_box.value} | Gene: {gene_box.value} | {conseq_box.value}"
        render_variant_result(label, proba, details)

predict_btn.on_click(on_predict_variant)

display(widgets.HTML("<b style='font-size:15px;'>Variant Details</b>"))
display(widgets.HBox([position_box, ref_box, alt_box]))
display(widgets.HBox([gene_box, conseq_box]))
display(widgets.HTML("<br><b style='font-size:15px;'>Quality Metrics</b>"))
display(depth_box, altfreq_box, quality_box, gnomad_box, conservation_box)
display(predict_btn)
display(out2)


HTML(value="<b style='font-size:15px;'>Variant Details</b>")

HTML(value="<br><b style='font-size:15px;'>Quality Metrics</b>")

IntSlider(value=40, description='Read Depth', layout=Layout(width='400px'), max=200, min=8, style=SliderStyle(…

FloatSlider(value=0.45, description='Allele Freq (VAF)', layout=Layout(width='400px'), max=1.0, step=0.01, sty…

FloatSlider(value=60.0, description='Quality Score', layout=Layout(width='400px'), step=0.5, style=SliderStyle…

FloatSlider(value=0.001, description='gnomAD Freq', layout=Layout(width='400px'), max=0.5, readout_format='.3f…

FloatSlider(value=0.8, description='Conservation Score', layout=Layout(width='400px'), max=1.0, step=0.01, sty…

Button(button_style='success', description='🧬 Variant Predict Karein', layout=Layout(height='42px', width='260…

Output()